# Feature Engineering

### Context
This notebook constructs and prunes the candidate feature set for the 12:00 PM operational forecasting pipeline. It transforms morning grid telemetry, observed surface weather, and afternoon Numerical Weather Prediction (HRRR) forecasts into a leak-free, 22-feature table aligned with the 4-dimensional ramp target vector.

### Decisions Made
- **Strict 12:00 PM Cutoff**: To match control-room conditions, all observed grid and weather features are restricted to the morning window (`00:00` to `12:00` local time), paired only with forward NOAA HRRR forecast leads (`f01`–`f09`).
- **Chronological Partition Before Screening**: I split the 2021–2025 development data at `2024-07-01` into training (`< 2024-07-01`, 559 days / 59.5%) and validation (`2024-07-01` to `2025-12-31`, 380 days / 40.5%) before running any correlation or importance checks, reserving 2026 (191 days) as a held-out test set (49.5% / 33.6% / 16.9% overall). This prevents selection leakage from validation distributions into the feature set.
- **Spatial Multicollinearity Pruning**: Spearman correlation analysis across the 4 regional centroids (BOS, ORH, BDL, BGR) showed extreme spatial redundancy (`|r| > 0.85`) in station temperatures and cloud layers. I retained Worcester (ORH) as the central Massachusetts solar corridor anchor and Boston (BOS) for the demand center, dropping 14 redundant features.
- **Quantile Gain Pruning**: I evaluated the remaining 27 features using LightGBM split gain under the project's operational quantile objectives ($\alpha = 0.85$ for $\Delta MW_{\text{norm}}$ and $S_{\max}$, $\alpha = 0.15$ for $t_{\text{start}}$, and $\alpha = 0.50$ for $\Delta t$), removing 5 low-gain peripheral features to reach a compact 22-feature set.

### Output Artifacts
- `data/processed/features_selected.parquet`: Final 22-feature table paired with daily ramp targets.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pvlib

from src.plotting import apply_style

pd.options.mode.copy_on_write = True
pd.options.future.infer_string = True

apply_style()

DATA_DIR = Path("data")
CENTROIDS = ["BOS", "ORH", "BDL", "BGR"]

## Data Loading

I load the daily ramp targets alongside the three raw input streams—5-minute grid telemetry, hourly surface weather observations, and NOAA HRRR forecasts—and immediately filter every table to the qualifying ramp dates so non-ramp days never enter feature calculations.

In [2]:
targets_df = pd.read_parquet(DATA_DIR / "processed/daily_ramp_targets.parquet")
targets_df["date"] = pd.to_datetime(targets_df["date"]).dt.date
qualifying_dates = set(targets_df["date"])
print(f"Loaded {len(targets_df)} qualifying target days.")

telem_df = pd.read_parquet(DATA_DIR / "processed/telemetry_5min.parquet")
telem_df["date"] = telem_df["interval_start_local"].dt.date
telem_df = telem_df[telem_df["date"].isin(qualifying_dates)].copy()
print(f"Loaded {len(telem_df):,} 5-minute telemetry intervals.")

weather_files = sorted((DATA_DIR / "raw/weather_observed").glob("weather_observations*.parquet"))
obs_df = pd.concat([pd.read_parquet(f) for f in weather_files], ignore_index=True)
obs_df["valid_time_local"] = obs_df["date"].dt.tz_convert("US/Eastern")
obs_df["date_local"] = obs_df["valid_time_local"].dt.date
obs_df = obs_df[obs_df["date_local"].isin(qualifying_dates)].copy()
print(f"Loaded {len(obs_df):,} weather observation rows.")

hrrr_files = sorted((DATA_DIR / "raw/weather_forecast").glob("hrrr_forecasts_*.parquet"))
hrrr_df = pd.concat([pd.read_parquet(f) for f in hrrr_files], ignore_index=True)
hrrr_df["forecast_date"] = pd.to_datetime(hrrr_df["forecast_date"]).dt.date
hrrr_df = hrrr_df[hrrr_df["forecast_date"].isin(qualifying_dates)].copy()

# Convert temperature from Kelvin to Celsius to match Open-Meteo observations
hrrr_df["tmp_2m_c"] = hrrr_df["tmp_2m"].astype(np.float32) - 273.15
print(f"Loaded {len(hrrr_df):,} HRRR forecast rows across {len(hrrr_files)} annual files.")

Loaded 1130 qualifying target days.
Loaded 325,428 5-minute telemetry intervals.
Loaded 108,476 weather observation rows.
Loaded 40,644 HRRR forecast rows across 6 annual files.


All four streams are now aligned on local `US/Eastern` dates for the qualifying ramp days, with HRRR temperatures converted to °C to match the observed weather archive.

## Morning Telemetry Features

I extract 13 pre-noon features strictly over the morning observation window (`00:00` to `12:00` local time). These capture morning load trajectory, midday solar penetration, clear-sky index ($k_t$), and total installed capacity before the 12:00 PM cutoff.

In [3]:
def extract_telemetry_features(df_day: pd.DataFrame) -> dict:
    """Extract pre-noon demand, solar, and net load features for a single day."""
   

    morning = df_day[df_day["interval_start_local"].dt.hour < 12].copy()
    if morning.empty:
        return {}
    
    noon_row = df_day[df_day["interval_start_local"].dt.hour == 12].iloc[0] 

    ################Grid Demand Features###########
    gross_noon = float(noon_row["gross_demand"])
    gross_peak_am = float(morning["gross_demand"].max())
    
    #Slope: 10:00 AM to 12:00 PM
    h10 = morning[morning["interval_start_local"].dt.hour >= 10]
    gross_h10 = float(h10.iloc[0]["gross_demand"]) if not h10.empty else gross_noon
    gross_slope_am = (gross_noon - gross_h10) / 2.0  # MW per hour
    
    # Volatility: standard deviation from 11:00 AM to 12:00 PM
    h11 = morning[morning["interval_start_local"].dt.hour >= 11]
    gross_std_1h = float(h11["gross_demand"].std()) if len(h11) > 1 else 0.0
    
    ####Solar Telemetry Features##########
    solar_cf_noon = float(noon_row["solar_capacity_factor"])
    h8_12 = morning[morning["interval_start_local"].dt.hour >= 8]
    solar_cf_max_am = float(h8_12["solar_capacity_factor"].max()) if not h8_12.empty else solar_cf_noon
    solar_cf_decay = float(solar_cf_noon / solar_cf_max_am) if solar_cf_max_am > 0 else 1.0
    
    total_solar_noon = float(noon_row["btm_solar"] + noon_row["utility_solar"])
    solar_pen_noon = float(total_solar_noon / gross_noon) if gross_noon > 0 else 0.0
    
    # Clear-Sky Index k_t = solar_capacity_factor / (clearsky_ghi / 1000)
    expected_clear_cf = (h8_12["clearsky_ghi"] / 1000.0).clip(lower=0.01)
    kt_series = (h8_12["solar_capacity_factor"] / expected_clear_cf).clip(0.0, 1.5)
    # 3-hour window from 09:00 to 12:00
    h9_12_mask = h8_12["interval_start_local"].dt.hour >= 9
    kt_3h = kt_series[h9_12_mask]
    solar_kt_mean_3h = float(kt_3h.mean()) if not kt_3h.empty else 0.0
    solar_kt_std_3h = float(kt_3h.std()) if len(kt_3h) > 1 else 0.0
    
    # ###### Net Load Position ##################
    net_load_noon = float(noon_row["net_load"])
    net_load_trough_am = float(morning[morning["interval_start_local"].dt.hour >= 8]["net_load"].min())
    
    ############ Solar Geometry & Capacity baselines ###########
    solar_zenith_noon = float(noon_row["solar_zenith"])
    total_solar_cap = float(noon_row["TOTAL_SOLAR_CAPACITY_MW"])
    return {
        "date": df_day["date"].iloc[0],
        "gross_demand_noon": gross_noon,
        "gross_demand_peak_am": gross_peak_am,
        "gross_demand_slope_am": gross_slope_am,
        "gross_demand_std_1h": gross_std_1h,
        "solar_cf_noon": solar_cf_noon,
        "solar_cf_decay_noon": solar_cf_decay,
        "solar_penetration_noon": solar_pen_noon,
        "solar_kt_mean_3h": solar_kt_mean_3h,
        "solar_kt_std_3h": solar_kt_std_3h,
        "net_load_noon": net_load_noon,
        "net_load_trough_am": net_load_trough_am,
        "solar_zenith_noon": solar_zenith_noon,
        "total_solar_capacity_mw": total_solar_cap,
    }

# Test on a sample day
sample_date = sorted(list(qualifying_dates))[10]
sample_telem = telem_df[telem_df["date"] == sample_date]
sample_feats = extract_telemetry_features(sample_telem)
pd.Series(sample_feats)

date                        2021-03-21
gross_demand_noon            12254.358
gross_demand_peak_am         12306.001
gross_demand_slope_am            5.885
gross_demand_std_1h          19.098207
solar_cf_noon                 0.755287
solar_cf_decay_noon           1.004767
solar_penetration_noon        0.257976
solar_kt_mean_3h              1.025898
solar_kt_std_3h               0.009502
net_load_noon              8915.691333
net_load_trough_am            8921.284
solar_zenith_noon            43.495635
total_solar_capacity_mw     4185.60274
dtype: object

## Observed Weather Features

To capture pre-noon atmospheric conditions across New England, I extract 12:00 PM temperature, 3-hour morning warming rate (`09:00` to `12:00`), and mean morning cloud cover across all four regional centroids (`BOS`, `ORH`, `BDL`, `BGR`).

In [4]:
def extract_observed_weather_features(df_obs_all: pd.DataFrame, target_dates: set) -> pd.DataFrame:
    """Extract observed weather metrics for all 4 centroids."""
    # Filter to 09:00 to 12:00 local time
    df_morning = df_obs_all[
        (df_obs_all["date_local"].isin(target_dates)) &
        (df_obs_all["valid_time_local"].dt.hour >= 9) &
        (df_obs_all["valid_time_local"].dt.hour <= 12)
    ].copy()
    
    results = []
    for date_val, group in df_morning.groupby("date_local"):
        row_dict = {"date": date_val}
        for stid in CENTROIDS:
            st_data = group[group["point_stid"] == stid].sort_values("valid_time_local")
            if st_data.empty:
                continue
            
            # 12:00 PM temperature
            t_noon_match = st_data[st_data["valid_time_local"].dt.hour == 12]
            t_noon = float(t_noon_match.iloc[0]["temperature_2m"]) if not t_noon_match.empty else float(st_data.iloc[-1]["temperature_2m"])
            row_dict[f"temp_noon_{stid}"] = t_noon
            
            # 3-hour morning warming: T(12:00) - T(09:00)
            t_09_match = st_data[st_data["valid_time_local"].dt.hour == 9]
            t_09 = float(t_09_match.iloc[0]["temperature_2m"]) if not t_09_match.empty else t_noon
            row_dict[f"temp_trend_am_{stid}"] = t_noon - t_09
            
            # Mean morning cloud cover (09:00 to 12:00)
            row_dict[f"cloud_cover_am_{stid}"] = float(st_data["cloud_cover"].mean())
            
        results.append(row_dict)
    return pd.DataFrame(results)

obs_features_df = extract_observed_weather_features(obs_df, qualifying_dates)
print(f"Extracted observed weather features for {len(obs_features_df)} days. Columns: {len(obs_features_df.columns) - 1}")
obs_features_df.head(3)

Extracted observed weather features for 1130 days. Columns: 12


,date,temp_noon_BOS,temp_trend_am_BOS,cloud_cover_am_BOS,temp_noon_ORH,temp_trend_am_ORH,cloud_cover_am_ORH,temp_noon_BDL,temp_trend_am_BDL,cloud_cover_am_BDL,temp_noon_BGR,temp_trend_am_BGR,cloud_cover_am_BGR
0,2021-03-06,-0.4525,3.80,0.00,-2.10,3.50,0.00,-0.9,2.80,9.25,-2.95,5.25,0.00
1,2021-03-07,0.1475,4.05,0.00,-1.05,4.25,6.75,1.0,4.35,3.75,-3.65,3.70,33.25
2,2021-03-08,2.0475,3.90,5.25,1.35,4.80,0.00,2.5,5.00,0.00,0.10,5.05,0.00


## Weather Forecast Features

I anchor the afternoon forecast window to astronomical sunset at Worcester (`ORH`), the center of the Massachusetts solar corridor, using `pvlib`. Condensing HRRR forecast leads (`f01`–`f09`) into afternoon temperature drop, mean afternoon cloud cover, and sunset-window cloud cover produces 13 forward-looking features.

In [15]:
import pvlib

# Precalculate astronomical sunset for each date at Worcester (ORH) - Solar Center
orh_lat, orh_lon = 42.2626, -71.8023
dates_dt = pd.to_datetime(list(qualifying_dates))
sun_times = pvlib.solarposition.sun_rise_set_transit_spa(
    dates_dt.tz_localize("US/Eastern"),
    orh_lat,
    orh_lon,
)
sunset_dict = dict(zip(dates_dt.date, sun_times["sunset"].dt.tz_convert("US/Eastern")))

def extract_forecast_features(df_hrrr_all: pd.DataFrame, sunset_map: dict) -> pd.DataFrame:
    """Extract HRRR forecast features for all 4 centroids."""
    results = []
    for date_val, group in df_hrrr_all.groupby("forecast_date"):
        if date_val not in sunset_map:
            continue
        sunset_time = sunset_map[date_val]
        sunset_hour = sunset_time.hour
        
        # Minutes from 12:00 PM to sunset
        noon_local = pd.Timestamp(f"{date_val} 12:00", tz="US/Eastern")
        sunset_min_offset = (sunset_time - noon_local).total_seconds() / 60.0
        
        row_dict = {"date": date_val,
            "sunset_minute_offset": sunset_min_offset}
        
        for stid in CENTROIDS:
            st_fcst = group[group["point_stid"] == stid].sort_values("lead_hour")
            if st_fcst.empty:
                continue
                
            # Temperature drop to sunset: T(sunset_hour) - T(f01)
            t_sunset_match = st_fcst[st_fcst["valid_time_local"].dt.hour == sunset_hour]
            t_sunset = float(t_sunset_match.iloc[0]["tmp_2m_c"]) if not t_sunset_match.empty else float(st_fcst.iloc[-1]["tmp_2m_c"])
            t_noon_fcst = float(st_fcst.iloc[0]["tmp_2m_c"])  # f01 (1:00 PM)
            row_dict[f"temp_drop_afternoon_{stid}"] = t_sunset - t_noon_fcst
            
            # Mean afternoon cloud cover (all available afternoon leads f01-f08)
            row_dict[f"cloud_cover_afternoon_{stid}"] = float(st_fcst["tcdc"].mean())
            
            # Sunset window cloud cover (sunset_hour ± 1 hour)
            sunset_window = st_fcst[
                (st_fcst["valid_time_local"].dt.hour >= sunset_hour - 1) &
                (st_fcst["valid_time_local"].dt.hour <= sunset_hour + 1)
            ]
            row_dict[f"cloud_cover_sunset_{stid}"] = float(sunset_window["tcdc"].mean()) if not sunset_window.empty else float(st_fcst.iloc[-1]["tcdc"])
            
        results.append(row_dict)
    return pd.DataFrame(results)

fcst_features_df = extract_forecast_features(hrrr_df, sunset_dict)
print(f"Extracted forecast features for {len(fcst_features_df)} days. Columns: {len(fcst_features_df.columns) - 1}")
fcst_features_df.head(3)

Extracted forecast features for 1129 days. Columns: 13


,date,sunset_minute_offset,temp_drop_afternoon_BOS,cloud_cover_afternoon_BOS,cloud_cover_sunset_BOS,temp_drop_afternoon_ORH,cloud_cover_afternoon_ORH,cloud_cover_sunset_ORH,temp_drop_afternoon_BDL,cloud_cover_afternoon_BDL,cloud_cover_sunset_BDL,temp_drop_afternoon_BGR,cloud_cover_afternoon_BGR,cloud_cover_sunset_BGR
0,2021-03-06,343.494533,-0.75,63.666668,86.333336,-1.50,76.000000,99.666664,-1.25,79.333336,100.000000,-1.25,52.222221,100.000000
1,2021-03-07,344.682115,0.50,1.555556,1.000000,0.50,0.444444,0.666667,-0.50,0.888889,0.333333,-0.50,1.888889,1.333333
2,2021-03-08,345.865741,-0.25,34.111111,59.000000,0.25,23.666666,40.000000,1.00,8.111111,8.333333,-1.25,0.333333,0.000000


## Feature Matrix Assembly

I merge the telemetry, observed weather, and forecast tables with cyclical calendar features (`day_of_year_sin`, `day_of_year_cos`, `is_weekend`). To prevent selection leakage from validation distributions, I partition the 2021–2025 development set at `2024-07-01` before running any correlation or importance screening.

In [42]:
# Telemetry Features
telem_rows = []
for date_val, g in telem_df.groupby("date"):
    telem_rows.append(extract_telemetry_features(g))
telem_features_df = pd.DataFrame(telem_rows)

# Merge all feature tables
all_features_all_days_df = telem_features_df.merge(obs_features_df, on="date", how="inner")
all_features_all_days_df = all_features_all_days_df.merge(fcst_features_df, on="date", how="inner")

# Add Calendar features
date_series = pd.to_datetime(all_features_all_days_df["date"])
doy = date_series.dt.dayofyear
all_features_all_days_df["day_of_year_sin"] = np.sin(2 * np.pi * doy / 365.25)
all_features_all_days_df["day_of_year_cos"] = np.cos(2 * np.pi * doy / 365.25)
all_features_all_days_df["is_weekend"] = (date_series.dt.dayofweek >= 5).astype(int)

# Merge target vector for downstream correlation & importance screening
target_cols = ["date", "t_start_min", "delta_mw_norm", "delta_t_min", "s_max"]
all_features_all_days_df["is_weekend"] = (date_series.dt.dayofweek >= 5).astype(int)
all_features_all_days_df= all_features_all_days_df.merge(targets_df[target_cols], on="date", how="inner")

# Filter to training dates
split_date = pd.to_datetime("2024-07-01").date()
test_date = pd.to_datetime("2025-12-31").date()

candidate_df = all_features_all_days_df[all_features_all_days_df["date"] < split_date].copy()
val_df = all_features_all_days_df[(all_features_all_days_df["date"] >= split_date) & (all_features_all_days_df['date'] <= test_date)].copy()

print(f"Training set:   {len(candidate_df)} days ({candidate_df['date'].min()} to {candidate_df['date'].max()})")
print(f"Validation set: {len(val_df)} days ({val_df['date'].min()} to {val_df['date'].max()})")

feature_cols = [c for c in candidate_df.columns if c not in set(target_cols)]
print(f"Candidate feature table shape: {candidate_df.shape}")

print(f"Qualifying days: {len(candidate_df)}, Features: {len(feature_cols)} (Ratio: {len(candidate_df)/len(feature_cols):.1f}:1)")



Training set:   559 days (2021-03-06 to 2024-06-29)
Validation set: 380 days (2024-07-01 to 2025-12-22)
Candidate feature table shape: (559, 46)
Qualifying days: 559, Features: 41 (Ratio: 13.6:1)


The assembled table yields 41 candidate features across 559 training days (`2021-03-06` to `2024-06-30`) and 380 validation days (`2024-07-01` to `2025-12-31`), giving a 13.6:1 training sample-to-feature ratio prior to pruning.

## Spatial Collinearity Screening

New England weather stations sit close enough together that temperature and cloud variables move in lockstep. I compute pairwise Spearman rank correlations strictly on the 559 training days (`candidate_df`) to identify redundant features with $|r| > 0.85$.

In [24]:
# Compute correlation matrix among features and targets
corr_matrix = candidate_df[feature_cols + ["delta_mw_norm", "s_max", "t_start_min", "delta_t_min"]].corr(method="spearman")

# Identify highly collinear feature pairs (|r| > 0.85)
high_corr = []
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        f1, f2 = feature_cols[i], feature_cols[j]
        r_val = corr_matrix.loc[f1, f2]
        if abs(r_val) > 0.85:
            high_corr.append((f1, f2, r_val))

high_corr_df = pd.DataFrame(high_corr, columns=["Feature 1", "Feature 2", "Spearman r"]).sort_values(by="Spearman r", ascending=False)
print(f"Found {len(high_corr_df)} highly correlated feature pairs (|r| > 0.85):")
high_corr_df.head(25)

Found 20 highly correlated feature pairs (|r| > 0.85):


,Feature 1,Feature 2,Spearman r
10,temp_noon_ORH,temp_noon_BDL,0.987734
3,net_load_noon,net_load_trough_am,0.985723
6,temp_noon_BOS,temp_noon_ORH,0.983961
0,gross_demand_noon,gross_demand_peak_am,0.971497
5,solar_zenith_noon,day_of_year_cos,0.971325
7,temp_noon_BOS,temp_noon_BDL,0.971020
11,temp_noon_ORH,temp_noon_BGR,0.949456
8,temp_noon_BOS,temp_noon_BGR,0.948121
12,temp_noon_BDL,temp_noon_BGR,0.940775
18,cloud_cover_afternoon_BDL,cloud_cover_sunset_BDL,0.905430


In [25]:
# Count how many times each feature appears in a collinear pair
collinear_features = pd.concat([high_corr_df["Feature 1"], high_corr_df["Feature 2"]]).value_counts()
print("Most frequent collinear features:")
print(collinear_features.head(10))


Most frequent collinear features:
temp_noon_BDL                4
temp_noon_BOS                4
temp_noon_ORH                3
cloud_cover_afternoon_ORH    3
temp_noon_BGR                3
solar_zenith_noon            2
sunset_minute_offset         2
gross_demand_slope_am        2
cloud_cover_afternoon_BOS    2
cloud_cover_afternoon_BDL    2
Name: count, dtype: int64


Station temperatures and sunset cloud layers across `BOS`, `ORH`, `BDL`, and `BGR` exhibit strong collinearity ($|r| > 0.85$). I retain Worcester (`ORH`) as the central Massachusetts solar anchor and Boston (`BOS`) for the demand hub, dropping 14 redundant station, cosine calendar, and trough features, then re-verify the remaining pairs.

In [ ]:
initial_drops = [
    "temp_noon_BDL",
    "temp_noon_BGR",
    "cloud_cover_afternoon_BOS",
    "cloud_cover_sunset_BOS",
    "solar_zenith_noon",
    "net_load_trough_am",
    "temp_noon_ORH",
    "gross_demand_peak_am",
    "day_of_year_cos",
    "cloud_cover_am_BOS",
    "cloud_cover_afternoon_BDL",
    "cloud_cover_sunset_BDL",
    "cloud_cover_sunset_ORH",
    "cloud_cover_sunset_BGR",
]

pruned_feature_cols = [c for c in feature_cols if c not in initial_drops]
print(f"Remaining candidate features: {len(pruned_feature_cols)} (down from {len(feature_cols)})")

corr_sub = candidate_df[pruned_feature_cols].corr(method="spearman")
remaining_high_corr = []
for i in range(len(pruned_feature_cols)):
    for j in range(i + 1, len(pruned_feature_cols)):
        f1, f2 = pruned_feature_cols[i], pruned_feature_cols[j]
        r_val = corr_sub.loc[f1, f2]
        if abs(r_val) > 0.85:
            remaining_high_corr.append((f1, f2, r_val))

remaining_corr_df = pd.DataFrame(
    remaining_high_corr, columns=["Feature 1", "Feature 2", "Spearman r"]
).sort_values(by="Spearman r", ascending=False)
print(f"Remaining collinear pairs (|r| > 0.85): {len(remaining_corr_df)}")
print(remaining_corr_df)

selected_feature_cols = pruned_feature_cols

## Gain Pruning

With spatial redundancy reduced to 27 features, I evaluate predictive signal on the 559 training days (`candidate_df`). I start by training a regularized baseline LightGBM regressor (`objective="regression"`) on each target to rank features by split gain.

In [68]:
target_cols = ["delta_mw_norm", "s_max", "t_start_min", "delta_t_min"]
X_train = candidate_df[selected_feature_cols]

importance_dict = {}
print("=== LightGBM Baseline Feature Importance - MSE (Split Gain) ===")
for target in target_cols:
    y_train = candidate_df[target]

    model = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=20,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=selected_feature_cols)
    imp_norm = (imp / imp.sum()) * 100
    importance_dict[target] = imp_norm

    print(f"\nTop 5 Most Important Features for {target}:")
    for feat, pct in imp_norm.sort_values(ascending=False).head(5).items():
        print(f"  {feat:30s}: {pct:5.1f}% of total gain")

Because symmetric MSE optimizes for the mean rather than operational reserve margins, I repeat the gain analysis using the target quantiles: $\alpha = 0.85$ for $\Delta MW_{\text{norm}}$ and $S_{\max}$, $\alpha = 0.15$ for early ramp onset ($t_{\text{start}}$), and $\alpha = 0.50$ for duration ($\Delta t$).

In [ ]:
quantiles = {
    "delta_mw_norm": 0.85,  # Conservative capacity protection
    "s_max": 0.85,          # Fast-peaker reserve protection
    "t_start_min": 0.15,    # Conservative early warning lead time
    "delta_t_min": 0.50,    # Median duration
}

importance_dict = {}
print("=== LightGBM Baseline Feature Importance - Quantile Loss (Split Gain) ===")
for target in target_cols:
    y_train = candidate_df[target]
    alpha = quantiles[target]

    model = lgb.LGBMRegressor(
        objective="quantile",
        alpha=alpha,
        n_estimators=100,
        learning_rate=0.05,
        num_leaves=15,
        min_child_samples=25,
        random_state=42,
        verbose=-1,
    )
    model.fit(X_train, y_train)

    imp = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=selected_feature_cols)
    imp_norm = (imp / imp.sum()) * 100
    importance_dict[target] = imp_norm

    print(f"\nTop 10 Most Important Features for {target} (alpha={alpha}):")
    for feat, pct in imp_norm.sort_values(ascending=False).head(10).items():
        print(f"  {feat:30s}: {pct:5.1f}% of total gain")

imp_df = pd.DataFrame(importance_dict)
imp_df["mean_gain_pct"] = imp_df.mean(axis=1)
imp_df["max_gain_pct"] = imp_df.max(axis=1)
imp_sorted = imp_df.sort_values(by="max_gain_pct", ascending=True)
print("\n=== 10 Least Informative Features (Lowest Max Gain Across All Targets) ===")
print(imp_sorted[["mean_gain_pct", "max_gain_pct"]].head(10).round(2))

## Feature Export

Across the quantile models, five peripheral Maine (`BGR`) and redundant Connecticut (`BDL`) or morning trend variables contribute negligible split gain. I drop these five features to lock in the final 22-feature table and export it for downstream modeling.

In [ ]:
importance_drops = [
    "temp_drop_afternoon_BGR",
    "cloud_cover_afternoon_BGR",
    "cloud_cover_am_BGR",
    "cloud_cover_am_BDL",
    "temp_trend_am_ORH",
]
all_pruned_drops = initial_drops + importance_drops
final_22_features = [c for c in feature_cols if c not in all_pruned_drops]

print(f"=== Final Feature Set: {len(final_22_features)} Features ===")
print(f"Sample-to-feature ratio on training set: {len(candidate_df) / len(final_22_features):.1f}:1\n")

categories = {
    "Grid Telemetry": [c for c in final_22_features if "demand" in c or "net_load" in c],
    "Solar Geometry": [c for c in final_22_features if "solar_cf" in c or "solar_pen" in c or "kt" in c],
    "Observed Weather": [c for c in final_22_features if "temp_noon" in c or "temp_trend" in c or "cloud_cover_am" in c],
    "Forecast Weather": [c for c in final_22_features if "drop" in c or "sunset_ORH" in c or "sunset_BDL" in c or "sunset_BGR" in c],
    "Calendar & Physics": [c for c in final_22_features if "sunset_minute" in c or "day_of_year" in c or "weekend" in c or "capacity" in c],
}
for cat, f_list in categories.items():
    print(f"{cat} ({len(f_list)}):")
    for f in f_list:
        print(f"  - {f}")
    print()

output_cols = ["date"] + final_22_features + target_cols
export_path = DATA_DIR / "processed/features_selected.parquet"
all_features_all_days_df[output_cols].to_parquet(export_path, index=False)
print(f"Saved final features to: {export_path.resolve()}")

Exported the locked 22-feature dataset to `data/processed/features_selected.parquet`. Pruning down to 22 features raises the training sample-to-feature ratio to 25.4:1 (559 training days), leaving a compact feature set ready for model benchmarking.